# LAB3 — Avaliador LLM-as-Judge

---

| Campo | Detalhe |
|---|---|
| **Aula** | Aula 8 — Avaliação e Observabilidade em RAG |
| **Lab** | LAB3 — Avaliador de Relevância com LLM-as-Judge |
| **Objetivo** | Implementar e calibrar um avaliador de relevância LLM-as-Judge que retorna scores 0–1 com threshold em 0.5 |
| **Duração estimada** | 45–60 minutos |
| **Pré-requisitos** | LAB1 e LAB2 concluídos; chave OpenAI API válida |

---

## Contexto Teórico

**LLM-as-Judge** é uma técnica de avaliação automática onde um modelo de linguagem age como árbitro, determinando se um documento recuperado é relevante para uma dada pergunta. Em vez de precisar de rótulos humanos para cada par (pergunta, documento), delegamos esse julgamento ao próprio LLM — com saída estruturada e score numérico.

**Por que usar saída estruturada?**  
Com `with_structured_output()`, forçamos o modelo a retornar um JSON com campos definidos (score + justificativa), eliminando a necessidade de parsing frágil de texto livre.

**Threshold de decisão:**  
O threshold (limiar) define a fronteira entre "relevante" e "não relevante". Um threshold de 0.5 significa que documentos com score ≥ 0.5 são considerados relevantes. Calibrar esse valor é crucial para equilibrar precisão e recall.

## 1. Instalação de Dependências e Configuração da API

In [ ]:
# Instalar as bibliotecas necessárias
# langchain: framework principal de orquestração
# langchain-openai: integração com modelos OpenAI
# pydantic: validação de dados e saída estruturada
%pip install langchain langchain-openai pydantic -q

In [ ]:
import os
from google.colab import userdata  # Para acessar secrets do Colab

# Configurar a chave da API OpenAI
# No Colab: vá em Secrets (ícone de chave) e adicione OPENAI_API_KEY
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✓ OPENAI_API_KEY carregada com sucesso")
except Exception:
    # Fallback: definir manualmente (não recomendado em produção)
    os.environ["OPENAI_API_KEY"] = "sk-..."  # Substitua pela sua chave
    print("⚠ OPENAI_API_KEY definida manualmente — use Secrets no Colab")

## 2. Schema Pydantic para Saída Estruturada

Usamos Pydantic para definir exatamente o formato de saída do avaliador. O modelo será forçado a retornar:
- `score`: float entre 0 e 1 (0 = irrelevante, 1 = totalmente relevante)
- `justificativa`: explicação em linguagem natural da decisão

In [ ]:
from pydantic import BaseModel, Field

# Definição do schema de saída do avaliador
class RelevanceScore(BaseModel):
    """Schema para avaliação de relevância entre pergunta e documento."""
    
    # Score de relevância: 0.0 = totalmente irrelevante, 1.0 = totalmente relevante
    score: float = Field(
        ge=0.0,  # maior ou igual a 0
        le=1.0,  # menor ou igual a 1
        description="Score de relevância entre 0 (irrelevante) e 1 (totalmente relevante)"
    )
    
    # Explicação da decisão do modelo
    justificativa: str = Field(
        description="Explicação concisa do por que o documento é ou não relevante para a pergunta"
    )

# Testar o schema com dados de exemplo
exemplo = RelevanceScore(score=0.85, justificativa="O documento trata diretamente do tema.")
print("Schema validado com sucesso:")
print(f"  score: {exemplo.score}")
print(f"  justificativa: {exemplo.justificativa}")

## 3. Implementação do Avaliador LLM-as-Judge

`with_structured_output()` é um método do LangChain que instrui o modelo a sempre retornar dados no formato do schema Pydantic fornecido. Internamente, ele usa function calling da OpenAI para garantir conformidade com o schema.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Inicializar o modelo gpt-4o-mini (custo-benefício ideal para avaliação em escala)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0  # Zero para máxima consistência nas avaliações
)

# Vincular o schema de saída estruturada ao modelo
# O modelo será forçado a retornar sempre no formato RelevanceScore
avaliador_estruturado = llm.with_structured_output(RelevanceScore)

# Prompt do sistema para o avaliador
# O prompt é cuidadosamente construído para contexto jurídico brasileiro
prompt_avaliador = ChatPromptTemplate.from_messages([
    ("system", """Você é um avaliador especializado em direito brasileiro.
Sua tarefa é avaliar se um documento é relevante para responder uma pergunta jurídica.

Critérios de avaliação:
- Score 0.0–0.2: Totalmente irrelevante (assunto completamente diferente)
- Score 0.2–0.4: Pouco relevante (relação superficial ou tangencial)
- Score 0.4–0.6: Moderadamente relevante (alguma relação mas não responde diretamente)
- Score 0.6–0.8: Bastante relevante (aborda o tema mas pode estar incompleto)
- Score 0.8–1.0: Totalmente relevante (responde diretamente a pergunta)

Seja objetivo e consistente. Forneça uma justificativa concisa."""),
    ("human", """Pergunta: {pergunta}

Documento: {documento}

Avalie a relevância do documento para responder a pergunta.""")
])

# Criar a chain: prompt → llm com saída estruturada
chain_avaliador = prompt_avaliador | avaliador_estruturado

print("✓ Avaliador LLM-as-Judge configurado com sucesso")
print(f"  Modelo: gpt-4o-mini")
print(f"  Temperatura: 0 (determinístico)")
print(f"  Formato de saída: RelevanceScore (score + justificativa)")

## 4. Conjunto de Teste — 10 Pares (Pergunta, Documento, Label)

Para calibrar o avaliador, precisamos de pares com labels conhecidos. Utilizamos exemplos do direito civil, penal e administrativo brasileiro:
- **5 pares relevantes** (score esperado ≥ 0.7): documento responde diretamente à pergunta
- **5 pares irrelevantes** (score esperado ≤ 0.3): documento não tem relação com a pergunta

In [ ]:
# Conjunto de teste com 10 pares (pergunta, documento, label_esperado)
# label_esperado: 1 = relevante, 0 = irrelevante
pares_teste = [
    # ===== PARES RELEVANTES (5) =====
    {
        "id": 1,
        "pergunta": "Qual é o prazo prescricional para ação de indenização por danos morais?",
        "documento": """O Código Civil brasileiro estabelece em seu art. 206, §3º, V, que a pretensão 
        de reparação civil prescreve em 3 (três) anos. O Superior Tribunal de Justiça consolidou 
        entendimento (Súmula 278) de que o termo inicial da prescrição em ação de indenização por 
        danos morais é a data em que o ofendido teve ciência do fato lesivo.""",
        "label_esperado": 1,
        "score_minimo": 0.7
    },
    {
        "id": 2,
        "pergunta": "O que configura flagrante delito no direito penal brasileiro?",
        "documento": """O Código de Processo Penal, em seu art. 302, define as hipóteses de flagrante 
        delito: I - está cometendo a infração penal; II - acaba de cometê-la; III - é perseguido, 
        logo após, pela autoridade, pelo ofendido ou por qualquer pessoa, em situação que faça 
        presumir ser autor da infração; IV - é encontrado, logo depois, com instrumentos, armas, 
        objetos ou papéis que façam presumir ser ele autor da infração.""",
        "label_esperado": 1,
        "score_minimo": 0.7
    },
    {
        "id": 3,
        "pergunta": "Quais são os atos de improbidade administrativa que importam enriquecimento ilícito?",
        "documento": """A Lei nº 8.429/1992 (Lei de Improbidade Administrativa), com alterações da 
        Lei nº 14.230/2021, define em seu art. 9º que constitui ato de improbidade que importa 
        enriquecimento ilícito auferir qualquer tipo de vantagem patrimonial indevida em razão do 
        exercício de cargo, mandato, função, emprego ou atividade nas entidades mencionadas no 
        art. 1º desta lei.""",
        "label_esperado": 1,
        "score_minimo": 0.7
    },
    {
        "id": 4,
        "pergunta": "A pessoa jurídica pode sofrer dano moral?",
        "documento": """O Superior Tribunal de Justiça consolidou jurisprudência por meio da Súmula 227, 
        estabelecendo que 'a pessoa jurídica pode sofrer dano moral'. O fundamento reside na 
        proteção à honra objetiva da empresa, ou seja, sua reputação e credibilidade perante o 
        mercado. O STJ entende que atos que abalam o nome, a marca ou a imagem comercial da 
        pessoa jurídica ensejam reparação por danos morais.""",
        "label_esperado": 1,
        "score_minimo": 0.7
    },
    {
        "id": 5,
        "pergunta": "O que é ato ilícito segundo o Código Civil?",
        "documento": """O art. 186 do Código Civil Brasileiro define: 'Aquele que, por ação ou omissão 
        voluntária, negligência ou imprudência, violar direito e causar dano a outrem, ainda que 
        exclusivamente moral, comete ato ilícito.' Complementarmente, o art. 187 estabelece o 
        abuso de direito como ato ilícito: 'Também comete ato ilícito o titular de um direito que, 
        ao exercê-lo, excede manifestamente os limites impostos pelo seu fim econômico ou social, 
        pela boa-fé ou pelos bons costumes.'""",
        "label_esperado": 1,
        "score_minimo": 0.7
    },
    
    # ===== PARES IRRELEVANTES (5) =====
    {
        "id": 6,
        "pergunta": "Qual é o prazo prescricional para ação de indenização por danos morais?",
        "documento": """A biodiversidade brasileira é reconhecida mundialmente. O Brasil abriga cerca 
        de 20% de todas as espécies do planeta, incluindo mais de 55.000 espécies de plantas com 
        sementes. A Amazônia, o Cerrado e a Mata Atlântica são considerados hotspots de 
        biodiversidade global, sendo objeto de políticas de preservação ambiental.""",
        "label_esperado": 0,
        "score_maximo": 0.3
    },
    {
        "id": 7,
        "pergunta": "O que configura flagrante delito no direito penal brasileiro?",
        "documento": """O mercado financeiro brasileiro é regulado pelo Banco Central do Brasil e pela 
        Comissão de Valores Mobiliários (CVM). As taxas de juros são definidas pelo Comitê de 
        Política Monetária (COPOM) nas reuniões periódicas. A taxa Selic é o principal instrumento 
        de política monetária do Banco Central para controle da inflação.""",
        "label_esperado": 0,
        "score_maximo": 0.3
    },
    {
        "id": 8,
        "pergunta": "Quais são os atos de improbidade administrativa que importam enriquecimento ilícito?",
        "documento": """O futebol brasileiro possui uma das histórias mais ricas do esporte mundial. 
        A seleção brasileira conquistou cinco títulos mundiais (1958, 1962, 1970, 1994 e 2002). 
        O Campeonato Brasileiro é disputado anualmente por 20 clubes no sistema de pontos corridos, 
        sendo considerado um dos campeonatos mais disputados do mundo.""",
        "label_esperado": 0,
        "score_maximo": 0.3
    },
    {
        "id": 9,
        "pergunta": "A pessoa jurídica pode sofrer dano moral?",
        "documento": """As receitas da culinária nordestina brasileira são ricas em sabores únicos. 
        O baião de dois é um prato típico feito com feijão e arroz, temperos e muitas vezes 
        acompanhado de carne de sol. A cozinha regional reflete a história e a cultura dos 
        diferentes povos que habitaram o nordeste brasileiro.""",
        "label_esperado": 0,
        "score_maximo": 0.3
    },
    {
        "id": 10,
        "pergunta": "O que é ato ilícito segundo o Código Civil?",
        "documento": """O sistema solar é composto por oito planetas que orbitam o Sol. A Terra é 
        o terceiro planeta em distância do Sol e o único conhecido a abrigar vida. A Lua é o 
        único satélite natural da Terra e influencia as marés dos oceanos através de sua 
        gravidade. O sistema solar tem aproximadamente 4,6 bilhões de anos.""",
        "label_esperado": 0,
        "score_maximo": 0.3
    }
]

print(f"✓ Conjunto de teste criado com {len(pares_teste)} pares")
print(f"  Pares relevantes (label=1): {sum(1 for p in pares_teste if p['label_esperado']==1)}")
print(f"  Pares irrelevantes (label=0): {sum(1 for p in pares_teste if p['label_esperado']==0)}")

## 5. Executar o Avaliador em Todos os 10 Pares

Agora rodamos o avaliador em cada par e coletamos os scores e justificativas.

In [ ]:
import time

# Lista para armazenar os resultados de cada avaliação
resultados = []

print("Executando avaliações...\n")
print("-" * 80)

for par in pares_teste:
    # Invocar o avaliador com o par (pergunta, documento)
    resultado = chain_avaliador.invoke({
        "pergunta": par["pergunta"],
        "documento": par["documento"]
    })
    
    # Determinar se o score bate com o label esperado (usando threshold 0.5)
    predicao = 1 if resultado.score >= 0.5 else 0
    acerto = predicao == par["label_esperado"]
    
    # Armazenar o resultado completo
    resultados.append({
        "id": par["id"],
        "pergunta": par["pergunta"][:60] + "...",  # Truncar para exibição
        "label_esperado": par["label_esperado"],
        "score": resultado.score,
        "predicao": predicao,
        "acerto": acerto,
        "justificativa": resultado.justificativa
    })
    
    # Exibir resultado formatado
    status = "✓" if acerto else "✗"
    categoria = "RELEVANTE" if par["label_esperado"] == 1 else "IRRELEVANTE"
    print(f"Par {par['id']:2d} [{categoria:11s}] | Score: {resultado.score:.3f} | {status}")
    print(f"         Justificativa: {resultado.justificativa[:100]}")
    print()
    
    # Pequena pausa para evitar rate limiting
    time.sleep(0.5)

print("-" * 80)
print(f"\n✓ Avaliação concluída: {len(resultados)} pares processados")

## 6. Calcular Métricas com Threshold = 0.5

Avaliamos a **accuracy** do threshold padrão (0.5) e a distribuição de scores por categoria.

In [ ]:
import pandas as pd
import numpy as np

# Criar DataFrame com os resultados
df = pd.DataFrame(resultados)

# ===== ACCURACY GERAL =====
accuracy = df["acerto"].mean()  # Proporção de acertos

# ===== DISTRIBUIÇÃO DE SCORES POR CATEGORIA =====
scores_relevantes = df[df["label_esperado"] == 1]["score"]
scores_irrelevantes = df[df["label_esperado"] == 0]["score"]

print("=" * 60)
print("MÉTRICAS COM THRESHOLD = 0.5")
print("=" * 60)
print(f"\nAccuracy geral: {accuracy:.1%} ({int(accuracy*len(df))}/{len(df)} acertos)")

print("\n--- Distribuição de Scores ---")
print(f"\nPares RELEVANTES (n={len(scores_relevantes)}):")
print(f"  Média:  {scores_relevantes.mean():.3f}")
print(f"  Mín:    {scores_relevantes.min():.3f}")
print(f"  Máx:    {scores_relevantes.max():.3f}")
print(f"  Desvio: {scores_relevantes.std():.3f}")

print(f"\nPares IRRELEVANTES (n={len(scores_irrelevantes)}):")
print(f"  Média:  {scores_irrelevantes.mean():.3f}")
print(f"  Mín:    {scores_irrelevantes.min():.3f}")
print(f"  Máx:    {scores_irrelevantes.max():.3f}")
print(f"  Desvio: {scores_irrelevantes.std():.3f}")

# Mostrar tabela completa de resultados
print("\n--- Tabela de Resultados ---")
tabela = df[["id", "label_esperado", "score", "predicao", "acerto"]].copy()
tabela.columns = ["ID", "Label Real", "Score LLM", "Predição", "Acerto"]
print(tabela.to_string(index=False))

## 7. Gráfico: Scatter Plot Score vs Label Esperado

Visualizamos a distribuição dos scores e a linha de threshold para identificar erros de classificação.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configurar o estilo do gráfico
plt.figure(figsize=(10, 6))

# Separar acertos e erros para cores diferentes
df_acertos = df[df["acerto"] == True]
df_erros = df[df["acerto"] == False]

# Plotar pontos dos pares relevantes
df_rel_acerto = df_acertos[df_acertos["label_esperado"] == 1]
df_irr_acerto = df_acertos[df_acertos["label_esperado"] == 0]
df_rel_erro = df_erros[df_erros["label_esperado"] == 1]
df_irr_erro = df_erros[df_erros["label_esperado"] == 0]

# Adicionar jitter horizontal para evitar sobreposição de pontos
jitter = 0.03

# Plotar acertos relevantes (verde, círculo)
plt.scatter(
    df_rel_acerto["label_esperado"] + np.random.uniform(-jitter, jitter, len(df_rel_acerto)),
    df_rel_acerto["score"],
    c="green", marker="o", s=150, label="Relevante — Acerto", zorder=5
)

# Plotar acertos irrelevantes (blue, círculo)
plt.scatter(
    df_irr_acerto["label_esperado"] + np.random.uniform(-jitter, jitter, len(df_irr_acerto)),
    df_irr_acerto["score"],
    c="steelblue", marker="o", s=150, label="Irrelevante — Acerto", zorder=5
)

# Plotar erros (vermelho, X)
if len(df_erros) > 0:
    plt.scatter(
        df_erros["label_esperado"] + np.random.uniform(-jitter, jitter, len(df_erros)),
        df_erros["score"],
        c="red", marker="X", s=200, label="Erro de classificação", zorder=6
    )

# Adicionar IDs aos pontos
for _, row in df.iterrows():
    plt.annotate(
        f"Par {int(row['id'])}",
        (row["label_esperado"], row["score"]),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=8,
        color="gray"
    )

# Linha de threshold em 0.5
plt.axhline(
    y=0.5, color="orange", linestyle="--", linewidth=2,
    label="Threshold = 0.5"
)

# Área de decisão
plt.axhspan(0.5, 1.05, alpha=0.05, color="green", label="Zona RELEVANTE")
plt.axhspan(-0.05, 0.5, alpha=0.05, color="red", label="Zona IRRELEVANTE")

# Formatação do gráfico
plt.xlabel("Label Esperado (0 = Irrelevante, 1 = Relevante)", fontsize=12)
plt.ylabel("Score do Avaliador LLM", fontsize=12)
plt.title("Avaliador LLM-as-Judge: Score vs Label Esperado\n(threshold = 0.5)", fontsize=14)
plt.xticks([0, 1], ["Irrelevante (0)", "Relevante (1)"], fontsize=11)
plt.ylim(-0.05, 1.05)
plt.xlim(-0.3, 1.3)
plt.legend(loc="center right", fontsize=9)
plt.grid(axis="y", alpha=0.3)

# Adicionar texto de accuracy
plt.text(0.02, 0.98, f"Accuracy: {accuracy:.1%}",
         transform=plt.gca().transAxes,
         fontsize=12, verticalalignment="top",
         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.savefig("lab3_scatter_score_label.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Gráfico salvo como lab3_scatter_score_label.png")

## 8. Análise de Thresholds: Precision e Recall

**Precision** (Precisão): De todos os documentos classificados como relevantes, quantos realmente são? Alta precisão = poucos falsos positivos.

**Recall** (Revocação): De todos os documentos realmente relevantes, quantos foram encontrados? Alto recall = poucos falsos negativos.

Testaremos thresholds de 0.3 a 0.7 para encontrar o ponto ótimo.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Lista de thresholds a testar
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

# Rótulos reais
y_true = df["label_esperado"].values
scores_llm = df["score"].values

# Calcular métricas para cada threshold
metricas_threshold = []

print("=" * 70)
print(f"{'Threshold':>10} | {'Precision':>10} | {'Recall':>10} | {'F1-Score':>10} | {'Accuracy':>10}")
print("-" * 70)

for t in thresholds:
    # Gerar predições com o threshold atual
    y_pred = (scores_llm >= t).astype(int)
    
    # Calcular métricas (zero_division=0 para evitar erro quando sem predições positivas)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = (y_pred == y_true).mean()
    
    metricas_threshold.append({
        "threshold": t,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "accuracy": acc
    })
    
    # Marcar o threshold padrão
    marker = " ← padrão" if t == 0.5 else ""
    print(f"{t:>10.1f} | {prec:>10.3f} | {rec:>10.3f} | {f1:>10.3f} | {acc:>10.1%}{marker}")

print("=" * 70)

# Criar DataFrame para análise
df_thresholds = pd.DataFrame(metricas_threshold)

# Identificar o threshold com melhor F1
melhor_idx = df_thresholds["f1"].idxmax()
melhor_threshold = df_thresholds.loc[melhor_idx]
print(f"\n→ Melhor threshold (maior F1): {melhor_threshold['threshold']:.1f}")
print(f"  F1-Score: {melhor_threshold['f1']:.3f}")
print(f"  Precision: {melhor_threshold['precision']:.3f}")
print(f"  Recall: {melhor_threshold['recall']:.3f}")

In [ ]:
# Gráfico de Precision, Recall e F1 por Threshold
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfico 1: Precision vs Recall por threshold ---
ax1.plot(thresholds, df_thresholds["precision"], "b-o", label="Precision", linewidth=2)
ax1.plot(thresholds, df_thresholds["recall"], "r-s", label="Recall", linewidth=2)
ax1.plot(thresholds, df_thresholds["f1"], "g-^", label="F1-Score", linewidth=2)
ax1.axvline(x=0.5, color="orange", linestyle="--", label="Threshold padrão (0.5)")
ax1.set_xlabel("Threshold", fontsize=12)
ax1.set_ylabel("Métrica", fontsize=12)
ax1.set_title("Precision, Recall e F1 por Threshold", fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1.1)
ax1.set_xticks(thresholds)

# --- Gráfico 2: Accuracy por threshold ---
bars = ax2.bar(thresholds, df_thresholds["accuracy"],
               color=["orange" if t == 0.5 else "steelblue" for t in thresholds],
               width=0.07, edgecolor="black")

# Adicionar valores nas barras
for bar, acc in zip(bars, df_thresholds["accuracy"]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{acc:.0%}", ha="center", va="bottom", fontsize=11)

ax2.set_xlabel("Threshold", fontsize=12)
ax2.set_ylabel("Accuracy", fontsize=12)
ax2.set_title("Accuracy por Threshold", fontsize=13)
ax2.set_ylim(0, 1.15)
ax2.set_xticks(thresholds)
ax2.grid(axis="y", alpha=0.3)

plt.suptitle("Análise de Thresholds — Avaliador LLM-as-Judge", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("lab3_threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Gráfico de análise de thresholds salvo")

## 9. Exercício Prático

> **Tarefa:** Adicione 3 pares de sua escolha ao conjunto de teste e recalcule as métricas.
> 
> Sugestões de temas para os novos pares:
> - Lei Maria da Penha e medidas protetivas
> - Habeas corpus no direito processual penal
> - Responsabilidade objetiva do Estado
> 
> Dica: Crie pares desafiadores — por exemplo, um documento que menciona superficialmente o tema mas não responde à pergunta (score esperado ~0.4–0.5).

In [ ]:
# ============================================================
# EXERCÍCIO: Adicione seus 3 pares aqui
# Siga o mesmo formato dos pares acima
# ============================================================

pares_adicionais = [
    {
        "id": 11,
        "pergunta": "ESCREVA SUA PERGUNTA AQUI",
        "documento": "ESCREVA O DOCUMENTO AQUI",
        "label_esperado": 1,  # 1 = relevante, 0 = irrelevante
    },
    {
        "id": 12,
        "pergunta": "ESCREVA SUA PERGUNTA AQUI",
        "documento": "ESCREVA O DOCUMENTO AQUI",
        "label_esperado": 0,
    },
    {
        "id": 13,
        "pergunta": "ESCREVA SUA PERGUNTA AQUI",
        "documento": "ESCREVA O DOCUMENTO AQUI",
        "label_esperado": 1,
    },
]

# Avaliar os novos pares
resultados_adicionais = []
print("Avaliando pares adicionais...\n")

for par in pares_adicionais:
    if "ESCREVA" in par["pergunta"]:
        print(f"⚠ Par {par['id']}: placeholder detectado — preencha com seus dados")
        continue
    
    resultado = chain_avaliador.invoke({
        "pergunta": par["pergunta"],
        "documento": par["documento"]
    })
    
    predicao = 1 if resultado.score >= 0.5 else 0
    acerto = predicao == par["label_esperado"]
    
    resultados_adicionais.append({
        "id": par["id"],
        "pergunta": par["pergunta"][:60] + "...",
        "label_esperado": par["label_esperado"],
        "score": resultado.score,
        "predicao": predicao,
        "acerto": acerto,
        "justificativa": resultado.justificativa
    })
    print(f"Par {par['id']}: Score={resultado.score:.3f} | Acerto={'✓' if acerto else '✗'}")

# Combinar todos os resultados e recalcular métricas
if resultados_adicionais:
    df_total = pd.concat([df, pd.DataFrame(resultados_adicionais)], ignore_index=True)
    acc_total = df_total["acerto"].mean()
    print(f"\nAccuracy total (13 pares): {acc_total:.1%}")
    print(f"Melhora em relação aos 10 pares originais: {(acc_total - accuracy)*100:+.1f} pp")
else:
    print("\nPreencha os pares adicionais para calcular as métricas combinadas.")

## Conclusão

### O que aprendemos neste lab:

1. **Saída estruturada com Pydantic** garante consistência e elimina parsing manual do texto do LLM
2. **Temperatura zero** é essencial para avaliadores — precisamos de julgamentos consistentes e reproduzíveis
3. **Threshold de 0.5** é um bom ponto de partida, mas a calibração depende do contexto de uso
4. **Trade-off Precision vs Recall**: thresholds mais altos aumentam precisão mas reduzem recall

### Perguntas de Reflexão:

1. Em um sistema de busca jurídica para advogados, você preferiria maximizar **precision** ou **recall**? Por quê?

2. O avaliador LLM pode ter **vieses**? Que tipos de vieses poderiam surgir no contexto jurídico brasileiro?

3. Como você validaria se o avaliador LLM é mais preciso que um avaliador baseado em embeddings (similaridade de cosseno)?

4. Qual seria o custo estimado de avaliar 10.000 pares com gpt-4o-mini? Isso é viável em produção?

---
*MBA em RAG & CAG Aplicados a Direito e Segurança Pública · Aula 8*